In [35]:
import sympy as sym
from sympy import *
import numpy as np
from tabulate import tabulate
import itertools
#following the Voseggard et al. paper for principal frame parameters JOURNAL OF MAGNETIC RESONANCE, Series A 122, 111 – 119 ( 1996 ) ARTICLE NO. 0186

In [36]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    D = np.diag(eigenvalues)
    print('Diagonalized Quadrupolar Tensor:\n', D, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    sorted_eigenvalues = sorted((eigenvalues - avg_tensor), key=abs)

    return sorted_eigenvalues, avg_tensor

In [37]:
def quadrupolar_parameters(Ispin, w0, D_coeff, E_coeff):

    if w0 < Ispin:
        raise ValueError('Error: Nuclear spin > Larmor frequency! [Check if the position for Ispin and w0 in input are correct]')

    whz = w0*10**6          #get Larmor frequency in Hz
    #factor q given in article
    q = (3-4*Ispin*(Ispin + 1))/(16*(whz))

    #Define symbol for quadrupolar tensor and force them to be real
    AzzmAyy_Q,AzzmAxx_Q,AyymAxx_Q,Ayz_Q,Axz_Q,Axy_Q = sym.symbols('AzzmAyy_Q,AzzmAxx_Q,AyymAxx_Q,Ayz_Q,Axz_Q,Axy_Q', real=True)

    #Variable for each equation set
    quad_tensor = [(AzzmAyy_Q, Ayz_Q), (AzzmAxx_Q, Axz_Q), (AyymAxx_Q, Axy_Q)]

    # List to hold solutions for quadrupolar tensor terms
    solutions_Q = []

    for i, (diagonal_diff, off_diagonal) in enumerate(quad_tensor):
        
        eq1 = sym.Eq(-((diagonal_diff)**2 - 4*(off_diagonal)**2)*((9*q/8)), D_coeff[i])
        eq2 = sym.Eq(off_diagonal*(diagonal_diff)*(9*q/2), E_coeff[i])

        # Solve the system
        solution = sym.solve([eq1, eq2], (diagonal_diff, off_diagonal))
        solutions_Q.append(solution)

    # Assign the solutions to the respective variables
    AzzmAyy_Q = [solutions_Q[0][0][0], solutions_Q[0][1][0]]
    Ayz_Q = [solutions_Q[0][0][1], solutions_Q[0][1][1]]
    AzzmAxx_Q = [solutions_Q[1][0][0], solutions_Q[1][1][0]]
    Axz_Q = [solutions_Q[1][0][1], solutions_Q[1][1][1]]
    AyymAxx_Q = [solutions_Q[2][0][0], solutions_Q[2][1][0]]
    Axy_Q = [solutions_Q[2][0][1], solutions_Q[2][1][1]]

   
    # Generate all combinations of AzzmAxx, AyymAxx, and AzzmAyy
    combinations = list(itertools.product(AzzmAxx_Q, AyymAxx_Q, AzzmAyy_Q))
    
    #Find Quadrupolar tensor diagonal elements

    Axx1 = []; Axx2 = []; Axx3 = []
    Ayy1 = []; Ayy2 = []; Ayy3 = []
    Azz1 = []; Azz2 = []; Azz3 = []

    combinations = list(itertools.product(AzzmAxx_Q, AyymAxx_Q, AzzmAyy_Q))

    # Initialize variables to track the best combination and minimum variation
    """
    Get the best combination for Axx1, Axx2, Axx3 with lowest standard deviation between them
    since there will be 8 combinations for Axx1, Axx2, Axx3
    """
    best_combination = None
    min_variation = float('inf')
    for (AzzmAxx_val, AyymAxx_val, AzzmAyy_val) in combinations:
        # Solution 1
        Axx1_val = (-(AzzmAxx_val + AyymAxx_val)/3) #using Axx + Ayy + Azz = 0
        Ayy1_val = Axx1_val + AyymAxx_val
        Azz1_val = Axx1_val + AzzmAxx_val

        #Save values
        Axx1.append(Axx1_val)
        Ayy1.append(Ayy1_val)
        Azz1.append(Azz1_val)

        # Solution 2
        Ayy2_val = -(AzzmAyy_val - AyymAxx_val) / 3
        Axx2_val = Ayy2_val - AyymAxx_val
        Azz2_val = Ayy2_val + AzzmAyy_val

        #Save values
        Axx2.append(Axx2_val)
        Ayy2.append(Ayy2_val)
        Azz2.append(Azz2_val)

        # Solution 3
        Azz3_val = (AzzmAxx_val + AzzmAyy_val) / 3
        Axx3_val = Azz3_val - AzzmAxx_val
        Ayy3_val = Azz3_val - AzzmAyy_val
        
        #Save values
        Axx3.append(Axx3_val)
        Ayy3.append(Ayy3_val)
        Azz3.append(Azz3_val)

        # Convert sympy Float to regular Python float for NumPy functions
        Axx1_val = float(Axx1_val)
        Axx2_val = float(Axx2_val)
        Axx3_val = float(Axx3_val)
        
        Ayy1_val = float(Ayy1_val)
        Ayy2_val = float(Ayy2_val)
        Ayy3_val = float(Ayy3_val)
        
        Azz1_val = float(Azz1_val)
        Azz2_val = float(Azz2_val)
        Azz3_val = float(Azz3_val)

        # Calculate variation (standard deviation) for Axx, Ayy, Azz
        variation_Axx = np.std([Axx1_val, Axx2_val, Axx3_val])
        variation_Ayy = np.std([Ayy1_val, Ayy2_val, Ayy3_val])
        variation_Azz = np.std([Azz1_val, Azz2_val, Azz3_val])

        total_variation = variation_Axx + variation_Ayy + variation_Azz

        # Update the best combination if the current one has less variation
        if total_variation < min_variation:
            min_variation = total_variation
            best_combination = (AzzmAxx_val, AyymAxx_val, AzzmAyy_val)
            best_Axx_Q = np.mean([Axx1_val, Axx2_val, Axx3_val])
            best_Ayy_Q = np.mean([Ayy1_val, Ayy2_val, Ayy3_val])
            best_Azz_Q = np.mean([Azz1_val, Azz2_val, Azz3_val])

        #Get index for off-diagonal elements        
    index_AzzmAxx = AzzmAxx_Q.index(best_combination[0])
    best_Axz_Q = Axz_Q[index_AzzmAxx]

    index_AyymAxx = AyymAxx_Q.index(best_combination[1])
    best_Axy_Q = Axy_Q[index_AyymAxx]

    index_AzzmAyy = AzzmAyy_Q.index(best_combination[2])
    best_Ayz_Q = Ayz_Q[index_AzzmAyy]

    #Quadrupolar Tensor in Tenon Frame
    Q_T = np.zeros((3,3))
    Q_T[0,0] = best_Axx_Q; Q_T[0,1] = best_Axy_Q; Q_T[0,2] = best_Axz_Q;
    Q_T[1,0] = best_Axy_Q; Q_T[1,1] = best_Ayy_Q; Q_T[1,2] = best_Ayz_Q;
    Q_T[2,0] = best_Axz_Q; Q_T[2,1] = best_Ayz_Q; Q_T[2,2] = best_Azz_Q;
    print('Quadrupolar tensor (tenon frame): \n', Q_T, '\n')
 
    sorted_eigenvalues, avg_tensor = sort_eigenvalues(Q_T)

    Vyy = (sorted_eigenvalues[0]+ avg_tensor)*(2*Ispin*(2*Ispin - 1)) 
    Vxx = (sorted_eigenvalues[1] + avg_tensor)*(2*Ispin*(2*Ispin - 1))
    Vzz = (sorted_eigenvalues[2] + avg_tensor)*(2*Ispin*(2*Ispin - 1)) 

    #Quadrupolar tensor parameters
    cq = Vzz/10**6
    etaq = (Vyy - Vxx)/Vzz

    return whz, q, best_Axx_Q, best_Ayy_Q, best_Azz_Q, best_Axy_Q, best_Ayz_Q, best_Axz_Q, cq, etaq

In [41]:
def csa_parameters(Ispin, w0, A_coeff, B_coeff, C_coeff):
    # Define symbol for CSA tensor and force them to be real
    Azz_s, Axx_s, Ayy_s, Ayz_s, Axz_s, Axy_s = sym.symbols('Azz_s,Axx_s,Ayy_s,Ayz_s,Axz_s,Axy_s', real=True)

    # Get the first six tensor values from quadrupolar_parameters
    values = quadrupolar_parameters(Ispin, w0, D_coeff, E_coeff)

    print(values)

    # Unpack only the required values
    whz, q, Axx_Q, Ayy_Q, Azz_Q, Axy_Q, Ayz_Q, Axz_Q = values[:8]

    # Variables for each equation
    cs_tensor = [
        (Ayy_s, Azz_s, Ayz_s),  # x -> Abb = Ayy; Agg = Azz; Abg = Ayz
        (Axx_s, Azz_s, Axz_s),  # y -> Abb = Axx; Agg = Azz; Abg = Axz
        (Axx_s, Ayy_s, Axy_s)   # z -> Abb = Axx; Agg = Ayy; Abg = Axy
    ]

    # Store variables in dictionary for access
    A = {
        'xx': Axx_Q, 'yy': Ayy_Q, 'zz': Azz_Q,
        'yz': Ayz_Q, 'zy': Ayz_Q,
        'xz': Axz_Q, 'zx': Axz_Q,
        'xy': Axy_Q, 'yx': Axy_Q,
    }

    # Define rotation tuple (a, b, g, bg, m)
    rotations = [
        ('x', 'y', 'z', 'yz', 1),  # a = x, b = y, g = z, m = 1
        ('y', 'x', 'z', 'xz', 1),  # a = y, b = x, g = z, m = 1
        ('z', 'x', 'y', 'xy', -1)  # a = z, b = x, g = y, m = -1
    ]

    # List to hold solutions
    solutions_cs = []

    for i, (Abb_s, Agg_s, Abg_s) in enumerate(cs_tensor):
        a, b, g, bg, m = rotations[i]
        eq1 = sym.Eq(
            (8 * A[a + a] * (A[b + b] + A[g + g] - A[a + a]) +
             16 * (A[a + b]**2 + A[a + g]**2) +
             5 * (A[b + b]**2 + A[g + g]**2) +
             28 * A[b + g]**2 - 18 * A[b + b] * A[g + g]) * (q / 8) - 
             0.5 * (Abb_s + Agg_s) * whz, A_coeff[i]
        )

        eq2 = sym.Eq(
            m * (2 * A[a + a] * (A[b + b] - A[g + g]) -
                 12 * (A[a + b]**2 - A[a + g]**2) -
                 A[b + b]**2 + A[g + g]**2) * (q / 2) - 
            0.5 * m * (Agg_s - Abb_s) * whz, B_coeff[i]
        )

        eq3 = sym.Eq(
            -m * (-2 * A[a + a] * A[b + g] +
                  12 * A[a + b] * A[a + g] + 
                  A[b + g] * (A[b + b] + A[g + g])) * q - 
            m * Abg_s * whz, C_coeff[i]
        )

        # Solve the system
        solution = sym.solve([eq1, eq2, eq3], (Abb_s, Agg_s, Abg_s))
        solutions_cs.append(solution)

    # Saving solutions
    Axx_s = np.mean([solutions_cs[1][Axx_s], solutions_cs[2][Axx_s]])
    Ayy_s = np.mean([solutions_cs[0][Ayy_s], solutions_cs[2][Ayy_s]])
    Azz_s = np.mean([solutions_cs[0][Azz_s], solutions_cs[1][Azz_s]])

    Ayz_s = solutions_cs[0][Ayz_s]
    Axz_s = solutions_cs[1][Axz_s]
    Axy_s = solutions_cs[2][Axy_s]

    # CSA tensor in tenon frame
    CS_T = np.zeros((3, 3))
    CS_T[0, 0] = Axx_s
    CS_T[0, 1] = Axy_s
    CS_T[0, 2] = Axz_s
    CS_T[1, 0] = Axy_s
    CS_T[1, 1] = Ayy_s
    CS_T[1, 2] = Ayz_s
    CS_T[2, 0] = Axz_s
    CS_T[2, 1] = Ayz_s
    CS_T[2, 2] = Azz_s

    print('Chemical Shift tensor (tenon frame): \n', CS_T)

    # Sort eigenvalues
    sorted_eigenvalues, avg_tensor = sort_eigenvalues(CS_T)

    csyy = -(sorted_eigenvalues[0] + avg_tensor)
    csxx = -(sorted_eigenvalues[1] + avg_tensor)
    cszz = -(sorted_eigenvalues[2] + avg_tensor)

    # CSA tensor parameters
    iso_cs = np.mean([cszz, csyy, csxx])  # converting Hz to ppm
    csa = cszz - iso_cs

    etas = (csyy - csxx) / csa

    return iso_cs, csa, etas


In [46]:
Ispin = 3/2
w0 = 192.55 #Larmor Frequency for 11B (MHz)
A_coeff = [-1.730215*10**3,-2.744504*10**3,-3.396561*10**3]
B_coeff = [1.251296*10**3,2.561384*10**3,-1.133846*10**3]
C_coeff = [3.573296*10**3,0.986618*10**3,-0.473004*10**3]
D_coeff = [-0.060956*10**3,-0.376258*10**3,-0.980912*10**3]
E_coeff = [-0.037878*10**3,-0.579924*10**3,-1.166911*10**3]

# def parameters(Ispin, w0, A_coeff, B_coeff, C_coeff, D_coeff, E_coeff):
quadrupolar_parameters(Ispin, w0, D_coeff, E_coeff)
csa_parameters(Ispin, w0, A_coeff, B_coeff, C_coeff)

    # return iso_cs, csa, etas

# whz = w0*10**6 #get Larmor frequency in Hz
# q = (3-4*Ispin*(Ispin + 1))/(16*(whz))
# A, B, C, D, E, F = solve_quadrupolar_tensor_equation(q, D_coeff, E_coeff)
# print(B)
# parameters(Ispin, w0, A_coeff, B_coeff, C_coeff, D_coeff, E_coeff)
# print(Q_T)

Quadrupolar tensor (tenon frame): 
 [[ 146208.94261904 -267333.19933213 -174507.29639701]
 [-267333.19933213  -94717.51332486   61530.65521205]
 [-174507.29639701   61530.65521205  -51491.42929418]] 

Diagonalized Quadrupolar Tensor:
 [[ 392392.82567951       0.               0.        ]
 [      0.         -278212.72231354       0.        ]
 [      0.               0.         -114180.10336597]] 

Quadrupolar tensor (tenon frame): 
 [[ 146208.94261904 -267333.19933213 -174507.29639701]
 [-267333.19933213  -94717.51332486   61530.65521205]
 [-174507.29639701   61530.65521205  -51491.42929418]] 

Diagonalized Quadrupolar Tensor:
 [[ 392392.82567951       0.               0.        ]
 [      0.         -278212.72231354       0.        ]
 [      0.               0.         -114180.10336597]] 

(192550000.0, -3.895092183848351e-09, 146208.94261903703, -94717.51332486019, -51491.429294176865, -267333.199332134, 61530.6552120510, -174507.296397013, 2.354356954077064, 0.41803164638272094)
Chemi

(-8.197131547580381e-06, 1.6622621307270817e-05, 0.37736117903599004)